In [1]:
# Plot the hydrogen-chain VQE benchmark results.
#
# Reads every {q}q_{l}l.json written by run_benchmark.py in RESULTS_DIR and
# saves one convergence figure per config to PLOT_DIR.
#
# Remove the matplotlib.use("Agg") line if you want to view figures
# interactively instead of only saving them (Agg is headless/cluster-safe).
print("Hello")
import os
import glob
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

RESULTS_DIR = "results"
PLOT_DIR = "plots"
DPI = 150


def mean_over_repeats(d, key_prefix):
    """Average all key_prefix{i} arrays (i = 1, 2, ...) found in dict d."""
    curves = []
    i = 1
    while f"{key_prefix}{i}" in d:
        curves.append(np.asarray(d[f"{key_prefix}{i}"], dtype=float))
        i += 1
    return np.mean(np.stack(curves), axis=0)


def collect_switches(d):
    """Gather the adaptive->constant switch steps across repeats."""
    switches, i = [], 1
    while f"loss{i}" in d:
        switches.append(d.get(f"weinstein_vs_constant{i}"))
        i += 1
    return switches


def lr_from_key(key):
    return float(key[len("lr_"):])


def reconstruct_regimes(n_steps, switches):
    """Per-step Weinstein (adaptive) / Constant labels from the switch step.

    switch is stored as (step + 1); the constant regime starts at index
    switch - 1. We use the median switch across repeats as representative of
    the averaged curve. If it never switched, the whole run is adaptive.
    """
    valid = [s for s in switches if s is not None]
    if not valid:
        return ["Weinstein"] * n_steps
    switch = int(round(np.median(valid)))
    return ["Weinstein" if i < switch - 1 else "Constant" for i in range(n_steps)]


def plot_config(path):
    with open(path) as f:
        r = json.load(f)

    num_qubits = r["num_qubits"]
    num_layers = r["num_layers"]
    gse = r["gse"]

    best_curve = mean_over_repeats(r["polyak_gd"], "best_loss")
    best_curve_qng = mean_over_repeats(r["polyak_qng"], "best_loss")

    sgd_lrs = sorted(r["sgd"].keys(), key=lr_from_key)
    adam_lrs = sorted(r["adam"].keys(), key=lr_from_key)
    qng_lrs = sorted(r["qng"].keys(), key=lr_from_key)
    sgd_curves = {k: mean_over_repeats(r["sgd"][k], "loss") for k in sgd_lrs}
    adam_curves = {k: mean_over_repeats(r["adam"][k], "loss") for k in adam_lrs}
    qng_curves = {k: mean_over_repeats(r["qng"][k], "loss") for k in qng_lrs}

    regimes = reconstruct_regimes(len(best_curve), collect_switches(r["polyak_gd"]))

    plt.figure(figsize=(16, 10))
    plt.plot(best_curve, marker='o', markersize=3, linestyle='-', color='black',
             linewidth=4, label="Polyak -> GD (best so far)")
    plt.plot(best_curve_qng, marker='d', markersize=3, linestyle='-', color='tab:cyan',
             linewidth=3, label="Polyak -> QNG (best so far)")

    styles = [':', '--', '-.', (0, (5, 1))]
    for j, k in enumerate(sgd_lrs):
        plt.plot(sgd_curves[k], marker='s', markersize=3, linestyle=styles[j % len(styles)],
                 color='tab:red', linewidth=2, label=f"GD (LR={lr_from_key(k)})")
    for j, k in enumerate(adam_lrs):
        plt.plot(adam_curves[k], marker='^', markersize=3, linestyle=styles[j % len(styles)],
                 color='tab:purple', linewidth=2, label=f"Adam (LR={lr_from_key(k)})")
    for j, k in enumerate(qng_lrs):
        plt.plot(qng_curves[k], marker='*', markersize=3, linestyle=styles[j % len(styles)],
                 color='tab:brown', linewidth=2, label=f"QNG (LR={lr_from_key(k)})")

    plt.axhline(gse, color='black', linestyle=':', linewidth=4, label="GSE")

    colors = {"Weinstein": "red", "Constant": "green"}
    labeled = set()
    start = 0
    for i in range(1, len(regimes) + 1):
        if i == len(regimes) or regimes[i] != regimes[start]:
            reg = regimes[start]
            plt.axvspan(start, i, color=colors[reg], alpha=0.15,
                        label=reg if reg not in labeled else None)
            labeled.add(reg)
            if i < len(regimes):
                plt.axvline(i, color='k', linestyle='--', linewidth=1.5)
            start = i

    plt.title(f"H-chain VQE: {num_qubits} qubits, {num_layers} layers (GSE = {gse:.4f})",
              fontsize=16)
    plt.xlabel("Optimization Step", fontsize=16)
    plt.ylabel("Expectation Value (Cost)", fontsize=16)
    plt.tick_params(labelsize=14)
    plt.legend(fontsize=10, framealpha=1, loc='upper right')
    plt.grid(True, linewidth=2.5)

    os.makedirs(PLOT_DIR, exist_ok=True)
    out = os.path.join(PLOT_DIR, f"{num_qubits}q_{num_layers}l.png")
    plt.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"saved -> {out}")


if __name__ == "__main__":
    files = glob.glob(os.path.join(RESULTS_DIR, "*.json"))
    files.sort(key=lambda p: json.load(open(p))["num_qubits"])
    if not files:
        print(f"No JSON files found in {RESULTS_DIR}/")
    for path in files:
        plot_config(path)
    print("Done.")

Hello
saved -> plots\4q_2l.png
saved -> plots\6q_3l.png
saved -> plots\8q_4l.png
saved -> plots\10q_5l.png
saved -> plots\12q_6l.png
saved -> plots\14q_7l.png
Done.


In [4]:
find . -name "*q_*l.json" 2>/dev/null

SyntaxError: invalid syntax (53633831.py, line 1)

In [2]:
find ~ -name "*q_*l.json" 2>/dev/null

SyntaxError: invalid syntax (40404370.py, line 1)